# DMRG 04: Observables and Diagnostics

This notebook shows how to measure physical quantities after a DMRG run. The focus is on quantities that are commonly used to interpret and validate DMRG results: ground-state energy, energy density, local expectation values, two-point correlations, entanglement entropy, truncation errors, and energy variance.


## Scope

The current implementation provides dense-contraction measurement utilities. This is a correctness baseline that is convenient for small systems and tutorials. For large production DMRG calculations, these APIs can later be backed by optimized MPS/MPO environment contractions without changing the user-facing workflow.

The excitation gap interface is present, but excited-state DMRG is not implemented yet.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_quantum_simulation():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation")


try:
    qs = _import_quantum_simulation()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    qs = _import_quantum_simulation()

print("Loaded quantum_simulation from", Path(qs.__file__).parent)


Loaded quantum_simulation from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation


In [2]:
import torch
import quantum_simulation as qs

from quantum_simulation import (
    DMRG,
    Ising1DMPOBuilder,
    MPS,
    Sweeps,
    energy_density,
    energy_variance,
    entanglement_entropy,
    excitation_gap,
    ground_state_energy,
    local_expectation,
    truncation_errors,
    two_point_correlation,
)

device = "cpu"  # Change to "cuda" after confirming your GPU setup.
dtype = torch.complex128
print(f"Using device={device}, dtype={dtype}")


Using device=cpu, dtype=torch.complex128


## Run a small DMRG calculation

We first optimize a small transverse-field Ising chain. The returned `energies` list contains one energy estimate per local two-site update. The `solver.history` dictionary stores additional diagnostics recorded during SVD truncation.


In [3]:
num_sites = 6
builder = Ising1DMPOBuilder(
    num_sites=num_sites,
    j_coupling=1.0,
    g_field=0.7,
    device=device,
    dtype=dtype,
)
mpo = builder.get_mpo()

initial_state = MPS(
    Nsites=num_sites,
    chid=2,
    initial_bond_dim=4,
    device=device,
    dtype=dtype,
    seed=2026,
)

sweeps = Sweeps(2)
sweeps.set_schedule(
    maxdim=[4, 8],
    noise=[1e-6, 0.0],
    krylov_dim=[6, 8],
    cutoff=[1e-10, 1e-12],
    reortho=[False, True],
    maxreortho=[1, 1],
)

solver = DMRG(initial_state, mpo, device=device, dtype=dtype, contract_backend="einsum")
energies, ground_state = solver(sweeps, dispon=1, maxit=2)
print(f"local updates: {len(energies)}")
print(ground_state)


Initializing MPS as a right-canonical random state...
Initialization complete. Ortho center at site 0.
[DMRG] Contraction backend resolved to 'einsum' on device='cpu'.
Sweep: 1 of 2, Energy: -2.327594413591, Bond dim: 4, Noise: 1.00e-06, Cutoff: 1.00e-10
Sweep: 2 of 2, Energy: -2.327594419162, Bond dim: 4, Noise: 0.00e+00, Cutoff: 1.00e-12
local updates: 20
MPS(Nsites=6, chid=2, ortho_center=0, bond_dims=[1, 2, 4, 6, 4, 2, 1])


## Ground-state energy and energy density

The final DMRG energy is a ground-state estimate for the chosen sweep schedule. Dividing by the number of sites gives the energy density, which is the more useful quantity for comparing different system sizes.


In [4]:
E0 = ground_state_energy(energies)
e0 = energy_density(E0, num_sites)

print(f"E0 = {E0:.12f}")
print(f"e0 = E0 / L = {e0:.12f}")


E0 = -2.327594419162
e0 = E0 / L = -0.387932403194


## Local expectation values

Local observables have the form `<O_i>`. Below we measure the spin operators `Sx` and `Sz` at every site. The Ising MPO used in this project is written in terms of spin-1/2 operators, so the eigenvalues of `Sz` are `+1/2` and `-1/2`.


In [5]:
# Sx = torch.tensor([[0.0, 0.5], [0.5, 0.0]], device=device, dtype=dtype)
# Sz = torch.tensor([[0.5, 0.0], [0.0, -0.5]], device=device, dtype=dtype)
Sx = qs.S_x(spin=0.5).to(device)
Sz = qs.S_z(spin=0.5).to(device)

sx_values = [local_expectation(ground_state, Sx, site).real.item() for site in range(num_sites)]
sz_values = [local_expectation(ground_state, Sz, site).real.item() for site in range(num_sites)]

print("site   <Sx_i>        <Sz_i>")
for site, (sx, sz) in enumerate(zip(sx_values, sz_values)):
    print(f"{site:>4d}  {sx: .10f}  {sz: .10f}")


site   <Sx_i>        <Sz_i>
   0   0.0000000000   0.4657078847
   1   0.0000000000   0.4348608005
   2   0.0000000000   0.4306110216
   3   0.0000000000   0.4306110216
   4   0.0000000000   0.4348608004
   5   0.0000000000   0.4657078847


## Two-point correlation functions

Two-point functions `<O_i O_j>` help distinguish short-range order, long-range order, and critical behavior. Here we measure `Sx-Sx` and `Sz-Sz` correlations relative to site 0.


In [6]:
print("j     <Sx_0 Sx_j>   <Sz_0 Sz_j>")
for j in range(num_sites):
    cxx = two_point_correlation(ground_state, Sx, 0, Sx, j).real.item()
    czz = two_point_correlation(ground_state, Sz, 0, Sz, j).real.item()
    print(f"{j:>1d}   {cxx: .10f}   {czz: .10f}")


j     <Sx_0 Sx_j>   <Sz_0 Sz_j>
0    0.2500000000    0.2500000000
1    0.0894911092    0.2300216336
2    0.0473205673    0.2029732314
3    0.0276641962    0.2010190594
4    0.0166551045    0.2026393878
5    0.0093931959    0.2169139055


## Entanglement entropy

The von Neumann entropy is computed from the Schmidt weights on each bond:

`S_E = -sum_alpha lambda_alpha^2 log(lambda_alpha^2)`.

Boundary bonds have zero entropy. Internal bonds reveal how entanglement is distributed across the chain.


In [7]:
entropies = entanglement_entropy(ground_state)
print("bond   S_E")
for bond, entropy in enumerate(entropies):
    print(f"{bond:>4d}  {entropy: .10f}")


bond   S_E
   0  -0.0000000000
   1   0.1493591142
   2   0.1680366931
   3   0.1715709183
   4   0.1680366931
   5   0.1493591142
   6  -0.0000000000


## Truncation errors and kept bond dimensions

The DMRG solver records the discarded singular weight after each two-site SVD. Small discarded weights indicate that the chosen maximum bond dimension is adequate for the current sweep schedule.


In [8]:
discarded = truncation_errors(solver)
print("update  sweep  direction  site  kept_dim  discarded_weight")
for update, value in enumerate(discarded):
    sweep = solver.history["sweeps"][update]
    direction = solver.history["directions"][update]
    site = solver.history["sites"][update]
    kept_dim = solver.history["bond_dims"][update]
    print(f"{update:>6d}  {sweep:>5d}  {direction:>9s}  {site:>4d}  {kept_dim:>8d}  {value: .3e}")


update  sweep  direction  site  kept_dim  discarded_weight
     0      1      right     0         2   0.000e+00
     1      1      right     1         4   0.000e+00
     2      1      right     2         4   4.156e-07
     3      1      right     3         4   0.000e+00
     4      1      right     4         2   0.000e+00
     5      1       left     4         2   0.000e+00
     6      1       left     3         4   0.000e+00
     7      1       left     2         4   2.837e-09
     8      1       left     1         4   0.000e+00
     9      1       left     0         2   0.000e+00
    10      2      right     0         2   0.000e+00
    11      2      right     1         4   0.000e+00
    12      2      right     2         6   1.148e-13
    13      2      right     3         4   0.000e+00
    14      2      right     4         2   0.000e+00
    15      2       left     4         2   0.000e+00
    16      2       left     3         4   0.000e+00
    17      2       left     2         6

## Energy variance

For an exact eigenstate, `Var(H) = <H^2> - <H>^2` is zero. In practice it is a useful convergence diagnostic. The current helper computes this with a dense MPO contraction, so keep the system size small.


In [9]:
variance = energy_variance(ground_state, mpo)
print(f"Var(H) = {variance.item():.12e}")


Var(H) = 1.349143019524e-12


## Excitation gap interface

The gap `Delta = E1 - E0` requires an excited-state solver or an independent way to estimate `E1`. The current module keeps the API slot explicit and raises `NotImplementedError` until excited-state DMRG is added.


In [10]:
try:
    excitation_gap()
except NotImplementedError as exc:
    print(exc)


Excitation gap requires excited-state DMRG or another E1 solver, which is not implemented yet.


## Takeaways

The post-DMRG workflow is now explicit: optimize the MPS, extract energy diagnostics, measure local and two-point observables, inspect entanglement, check discarded weights, and use energy variance as an eigenstate-quality diagnostic.
